# Astra research on Kaggle

Run the Hugging Face Forex experiment on Kaggle's CPU, then examine risk-gated, cost-aware historical results. **Accuracy and a profitable historical replay do not establish future profitability.** The reused development period and source-price assumptions remain explicit in the reports.

1. Upload `astra-kaggle.zip` as a **private Kaggle Dataset**, then add it to this notebook with **Add Input**.
2. Enable **Internet** in notebook settings for package installation and the first dataset download. A CPU session is sufficient.
3. Run the cells in order. Start with USDJPY; change the configuration below to include all four pairs.
4. Save a notebook version and download the result archive from the last cell or Kaggle's **Output** tab.

This notebook uses an isolated Python 3.12+ environment. All installations, tests, downloads, training and evaluation below run on Kaggle. No hosted run has been performed merely by exporting this notebook.


## 1. Find and stage the project

The default finds a single `astra-kaggle.zip` under `/kaggle/input`. Kaggle may expose the ZIP's contents directly; that is supported too. Keep `EXPORT-MANIFEST.json` with the exported files: this cell verifies the complete file inventory, sizes and SHA-256 hashes before installing or running project code. If several project inputs exist, set exactly one override below. Each execution creates a fresh project folder and preserves earlier results.


In [ ]:
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from hashlib import sha256
import json
import os
import shutil
import stat
import subprocess
import sys
import tempfile
import uuid
import zipfile

# Only needed when more than one Astra input is attached:
CODE_ZIP = None  # e.g. "/kaggle/input/astra-code/astra-kaggle.zip"
PROJECT_INPUT = None  # e.g. "/kaggle/input/astra-code"

INPUT_ROOT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
MANIFEST_NAME = "EXPORT-MANIFEST.json"
MAX_FILE_BYTES = 5 * 1024**2
MAX_SOURCE_BYTES = 30 * 1024**2
REQUIRED_SOURCE = {
    "pyproject.toml", "astra/__init__.py", "scripts/kaggle_run.py",
    "scripts/export_kaggle.py", "kaggle/requirements.txt",
    "kaggle/provenance/audit.json",
}
if not INPUT_ROOT.is_dir() or not WORKING.is_dir():
    raise RuntimeError("Run this notebook on Kaggle, not on your local machine.")
if CODE_ZIP and PROJECT_INPUT:
    raise ValueError("Choose either CODE_ZIP or PROJECT_INPUT.")

def is_project(path):
    return all((path / name).is_file() for name in REQUIRED_SOURCE | {MANIFEST_NAME})

def resolve_input(value):
    path = Path(value).resolve()
    if path != INPUT_ROOT and INPUT_ROOT not in path.parents:
        raise ValueError("Code input must be inside /kaggle/input.")
    return path

def manifest_inventory(payload):
    manifest = json.loads(payload)
    if manifest.get("schema") != "astra.kaggle.export.v1":
        raise ValueError("Missing or unrecognized Astra export manifest.")
    rows = manifest.get("files")
    if not isinstance(rows, list):
        raise ValueError("Export manifest has no file inventory.")
    inventory = {}
    for row in rows:
        name = row["path"]
        size = row["size_bytes"]
        digest = row["sha256"]
        if (not isinstance(name, str) or not name or "\\" in name or ":" in name
                or any(part in {"", ".", ".."} for part in name.split("/"))
                or PurePosixPath(name).is_absolute() or name == MANIFEST_NAME
                or name in inventory):
            raise ValueError(f"Invalid manifest path: {name!r}")
        if type(size) is not int or not 0 <= size <= MAX_FILE_BYTES:
            raise ValueError(f"Invalid manifest file size: {name}")
        if (not isinstance(digest, str) or len(digest) != 64
                or any(character not in "0123456789abcdef" for character in digest)):
            raise ValueError(f"Invalid manifest SHA-256: {name}")
        inventory[name] = row
    if REQUIRED_SOURCE - inventory.keys():
        raise ValueError("Export manifest is missing required Astra source files.")
    if sum(row["size_bytes"] for row in inventory.values()) > MAX_SOURCE_BYTES:
        raise ValueError("Expected a small source-code bundle, not raw market data.")
    return inventory

def verify_payload(name, payload, row):
    if len(payload) != row["size_bytes"] or sha256(payload).hexdigest() != row["sha256"]:
        raise ValueError(f"Source size/hash mismatch: {name}; re-export the project.")

def verify_project(directory):
    manifest_path = directory / MANIFEST_NAME
    if not manifest_path.is_file() or manifest_path.stat().st_size > MAX_FILE_BYTES:
        raise ValueError("Keep EXPORT-MANIFEST.json with the exported source files.")
    entries = list(directory.rglob("*"))
    if any(path.is_symlink() for path in entries):
        raise ValueError("Source input must not contain symlinks.")
    if any(not path.is_dir() and not path.is_file() for path in entries):
        raise ValueError("Source input must contain only regular files and directories.")
    inventory = manifest_inventory(manifest_path.read_bytes())
    actual = {path.relative_to(directory).as_posix() for path in entries if path.is_file()}
    expected = set(inventory) | {MANIFEST_NAME}
    if actual != expected:
        raise ValueError(
            f"Source inventory mismatch; missing={sorted(expected - actual)}, "
            f"extra={sorted(actual - expected)}"
        )
    for name, row in inventory.items():
        path = directory / name
        if path.stat().st_size != row["size_bytes"]:
            raise ValueError(f"Source size mismatch: {name}; re-export the project.")
        verify_payload(name, path.read_bytes(), row)
    return len(inventory)

def extract_source(archive_path, destination):
    # Check names and the complete manifest before extracting any source files.
    with zipfile.ZipFile(archive_path) as archive:
        members = archive.infolist()
        if len(members) > 10000 or sum(m.file_size for m in members) > MAX_SOURCE_BYTES:
            raise ValueError("Expected a small source-code bundle, not raw market data.")
        names = set()
        for member in members:
            name = member.filename
            relative = PurePosixPath(name)
            kind = stat.S_IFMT(member.external_attr >> 16)
            if (not name or "\\" in name or relative.is_absolute() or ".." in relative.parts
                    or any(":" in part for part in relative.parts)
                    or kind not in (0, stat.S_IFREG, stat.S_IFDIR)
                    or member.file_size > MAX_FILE_BYTES):
                raise ValueError(f"Unsafe ZIP member: {name!r}")
            target = (destination / Path(*relative.parts)).resolve()
            if target == destination or destination not in target.parents or target in names:
                raise ValueError(f"Unsafe or duplicate ZIP path: {name!r}")
            names.add(target)
        manifests = [
            member for member in members
            if not member.is_dir() and PurePosixPath(member.filename).name == MANIFEST_NAME
        ]
        if len(manifests) != 1:
            raise ValueError("ZIP must contain exactly one EXPORT-MANIFEST.json.")
        manifest_member = manifests[0]
        prefix = PurePosixPath(manifest_member.filename).parent
        inventory = manifest_inventory(archive.read(manifest_member))
        expected = {str(prefix / name): row for name, row in inventory.items()}
        actual = {member.filename for member in members if not member.is_dir()}
        if actual != set(expected) | {manifest_member.filename}:
            raise ValueError("ZIP inventory differs from its export manifest.")
        for name, row in expected.items():
            verify_payload(name, archive.read(name), row)
        archive.extractall(destination)

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
STAGING = WORKING / f"astra-project-{stamp}-{uuid.uuid4().hex[:8]}"
if PROJECT_INPUT:
    source = resolve_input(PROJECT_INPUT)
    if not is_project(source):
        raise ValueError(f"Not a complete exported Astra project: {source}")
    verify_project(source)
    shutil.copytree(source, STAGING)
else:
    archives = [resolve_input(CODE_ZIP)] if CODE_ZIP else sorted(
        INPUT_ROOT.rglob("astra-kaggle.zip")
    )
    if len(archives) > 1:
        raise ValueError(f"Several bundles found; set CODE_ZIP: {archives}")
    if archives:
        STAGING.mkdir()
        extract_source(archives[0], STAGING)
    else:
        candidates = sorted({
            item.parent for item in INPUT_ROOT.rglob(MANIFEST_NAME)
            if is_project(item.parent)
        })
        if len(candidates) != 1:
            raise ValueError(
                f"Attach astra-kaggle.zip or set PROJECT_INPUT; found {candidates}"
            )
        source = candidates[0]
        verify_project(source)
        shutil.copytree(source, STAGING)

projects = [STAGING] if is_project(STAGING) else sorted({
    item.parent for item in STAGING.rglob(MANIFEST_NAME)
    if is_project(item.parent)
})
if len(projects) != 1:
    raise ValueError(f"Expected one Astra project, found {projects}")
PROJECT = projects[0]
verified_files = verify_project(PROJECT)
print(f"Verified {verified_files} source files against the export manifest.")
print(f"Project: {PROJECT}")


## 2. Install an isolated runtime

Astra requires Python 3.12 or later. If the notebook kernel is older, this cell installs `uv` and downloads a managed Python 3.12 runtime. The notebook kernel itself stays unchanged; commands use the isolated runtime. Dependencies and installer caches live in the session's temporary directory, outside Kaggle's saved outputs. Internet must be enabled for this setup.


In [ ]:
# Keep dependencies out of Kaggle's saved /kaggle/working outputs.
RUNTIME_ROOT = Path(tempfile.mkdtemp(prefix="astra-runtime-", dir=tempfile.gettempdir()))
ENV_ROOT = RUNTIME_ROOT / "venv"
UV_BOOTSTRAP = RUNTIME_ROOT / "uv-bootstrap"
BOOTSTRAP_ENV = dict(os.environ)
BOOTSTRAP_ENV.update({
    "PIP_CACHE_DIR": str(RUNTIME_ROOT / "pip-cache"),
    "UV_CACHE_DIR": str(RUNTIME_ROOT / "uv-cache"),
    "UV_PYTHON_INSTALL_DIR": str(RUNTIME_ROOT / "python"),
    "PYTHONPATH": str(UV_BOOTSTRAP),
})
if sys.version_info >= (3, 12):
    subprocess.run(
        [sys.executable, "-m", "venv", str(ENV_ROOT)],
        env=BOOTSTRAP_ENV, check=True,
    )
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
        "--target", str(UV_BOOTSTRAP), "uv>=0.7,<1",
    ], env=BOOTSTRAP_ENV, check=True)
    uv_binary = next((path for path in (
        UV_BOOTSTRAP / "bin" / "uv", UV_BOOTSTRAP / "uv" / "uv",
    ) if path.is_file()), None)
    if uv_binary is None:
        raise RuntimeError("The temporary uv installation did not provide its executable.")
    subprocess.run([
        str(uv_binary), "venv", "--python", "3.12",
        "--managed-python", "--seed", str(ENV_ROOT),
    ], env=BOOTSTRAP_ENV, check=True)

PYTHON = ENV_ROOT / "bin" / "python"
RUN_ENV = dict(BOOTSTRAP_ENV)
RUN_ENV["PYTHONPATH"] = str(PROJECT)
RUN_ENV["PYTHONUNBUFFERED"] = "1"
RUN_ENV.pop("ASTRA_RUN_LIVE_TESTS", None)
print(f"Temporary runtime: {ENV_ROOT}")

def run_astra(arguments, log_path=None):
    # Stream both output channels into the notebook; keep no large memory buffer.
    command = [str(PYTHON), *map(str, arguments)]
    log = Path(log_path).open("w", encoding="utf-8") if log_path else None
    try:
        with subprocess.Popen(
            command, cwd=PROJECT, env=RUN_ENV,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log:
                        log.write(line)
                        log.flush()
                status = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.kill()
                process.wait()
                raise
        if status:
            raise subprocess.CalledProcessError(status, command)
    finally:
        if log:
            log.close()

run_astra(["--version"])
run_astra([
    "-m", "pip", "install", "--disable-pip-version-check",
    "-e", f"{PROJECT}[dev,ml]",
    "-r", PROJECT / "kaggle" / "requirements.txt",
])
run_astra(["-m", "scripts.kaggle_run", "--mode", "check"])


## 3. Choose the experiment

- `all`: train continuity-filtered models for the selected pairs, then evaluate USDJPY historical profitability.
- `train`: train and save models and diagnostics.
- `profitability`: evaluate an existing compatible training reference supplied through `REFERENCE_ROOT`.
- `check`: check Python and installed runtime dependencies without downloading or evaluating data.

USDJPY is the default research candidate. Use `["AUDUSD", "EURUSD", "GBPUSD", "USDJPY"]` to train all four. Profitability currently evaluates USDJPY only, so `all` must include USDJPY. Keep 400,000 rows for the established experiment size; smaller runs are exploratory and not directly comparable. Profitability uses the row limit saved in its training reference.

With `INPUT_CACHE = None`, the runner downloads the pinned public Hugging Face snapshot on Kaggle. To reuse your downloaded snapshot, attach its `hf-cache` directory as a private Kaggle input and set the path below. Attached inputs are read-only. Package installation still needs Internet unless you supply a separate offline environment.

The bundled sample audit corroborates BID prices and candle-open timestamps for sampled USDJPY rows. Training uses source timestamps plus one minute as close/availability times; the full cache remains unauthenticated. This changes the clock assumption from earlier local runs, which must not be treated as directly equivalent references.

The results directory contains reports, models and experiment journals. Downloaded source data is cached in temporary session storage, outside saved outputs and the result archive.


In [ ]:
MODE = "all"
PAIRS = ["USDJPY"]
MAX_ROWS = 400000
INPUT_CACHE = None  # e.g. "/kaggle/input/astra-hf-snapshot/hf-cache"
REFERENCE_ROOT = None  # Existing compatible continuity-training run for profitability mode.
RESULTS = WORKING / "astra-results"
RUN_TESTS = False  # Optional full software checks on Kaggle only.

if MODE not in {"all", "train", "profitability", "check"}:
    raise ValueError(f"Unsupported mode: {MODE}")
if not PAIRS or any(pair not in {"AUDUSD", "EURUSD", "GBPUSD", "USDJPY"} for pair in PAIRS):
    raise ValueError("Choose one or more supported currency pairs.")
if len(PAIRS) != len(set(PAIRS)):
    raise ValueError("Pairs must be unique.")
if not isinstance(MAX_ROWS, int) or MAX_ROWS < 300:
    raise ValueError("MAX_ROWS must be an integer of at least 300.")
if MODE == "all" and "USDJPY" not in PAIRS:
    raise ValueError("MODE='all' requires USDJPY for profitability evaluation.")
if MODE == "profitability" and not REFERENCE_ROOT:
    raise ValueError("Set REFERENCE_ROOT to a compatible training run, or use MODE='all'.")
RESULTS = Path(RESULTS).resolve()
if RESULTS == WORKING or WORKING not in RESULTS.parents:
    raise ValueError("Use a results subdirectory inside /kaggle/working.")
print(f"Mode={MODE}; pairs={PAIRS}; max rows per pair={MAX_ROWS:,}")
print(f"Results: {RESULTS}")


## 4. Optional software verification

Leave `RUN_TESTS = False` for the research run. Set it to `True` above to run the repository's pytest coverage suite and Ruff on Kaggle. Logs, JUnit results and coverage JSON will be included in the result archive. These are software checks using synthetic fixtures, not evidence of profitable trading.


In [ ]:
if RUN_TESTS:
    test_results = RESULTS / "verification" / f"{stamp}-{uuid.uuid4().hex[:8]}"
    test_results.mkdir(parents=True, exist_ok=False)
    run_astra([
        "-m", "pytest", "--cov=astra",
        f"--junitxml={test_results / 'pytest.xml'}",
        "--cov-report=term", f"--cov-report=json:{test_results / 'coverage.json'}",
    ], log_path=test_results / "pytest.log")
    run_astra([
        "-m", "ruff", "check", "astra", "scripts", "tests",
    ], log_path=test_results / "ruff.log")
else:
    print("Software checks skipped. Set RUN_TESTS=True to run them on Kaggle.")


## 5. Run

This step can take time, especially the first download. Leave the output visible for progress. If a command fails, the cell stops: inspect the error and any incomplete-run receipt before rerunning. Completed runs remain preserved in their own run directories.


In [ ]:
arguments = [
    "-m", "scripts.kaggle_run",
    "--mode", MODE,
    "--workspace", RESULTS,
    "--pairs", *PAIRS,
    "--max-rows", str(MAX_ROWS),
]
if INPUT_CACHE:
    arguments.extend(["--input-cache", str(INPUT_CACHE)])
if REFERENCE_ROOT:
    arguments.extend(["--reference-root", str(REFERENCE_ROOT)])
run_astra(arguments)


## 6. Read and download results

Read the generated reports and verification receipts before interpreting returns. Focus on net returns after costs, trade count, drawdown, uncertainty, and the effect of doubled costs. Development replay results cannot certify an untouched holdout, authenticate the full source, or establish actual bid/ask spreads and execution costs.

The archive below includes all saved runs in the results directory, including incomplete-run receipts and optional software-check logs. It excludes environments, source data and the uploaded code bundle. **Save Version** after a successful run so Kaggle keeps its outputs, then download the archive from this cell or the notebook's Output tab.


In [ ]:
from IPython.display import FileLink, Markdown, display

if not RESULTS.is_dir():
    print("No result directory yet; run an experiment before exporting results.")
else:
    blocked_parts = {".venv", "__pycache__", "hf-cache", "cache", ".cache"}
    raw_suffixes = {".arrow", ".parquet"}
    result_files = sorted(
        path for path in RESULTS.rglob("*")
        if path.is_file() and not path.is_symlink()
        and not blocked_parts.intersection(path.relative_to(RESULTS).parts)
        and path.suffix.lower() not in raw_suffixes
    )
    reports = [path for path in result_files if path.suffix.lower() == ".md"]
    print(f"Found {len(reports)} Markdown reports; {len(result_files)} result files.")
    for report in reports[:20]:
        display(FileLink(str(report)))
    latest_reports = {}
    for report in reports:
        if report.name not in latest_reports or report.stat().st_mtime > latest_reports[report.name].stat().st_mtime:
            latest_reports[report.name] = report
    for report in sorted(latest_reports.values(), key=lambda path: path.stat().st_mtime)[-2:]:
        display(Markdown(f"### {report.relative_to(RESULTS)}"))
        display(Markdown(report.read_text(encoding="utf-8")[:20000]))

    if result_files:
        archive_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive_path = WORKING / f"astra-results-{archive_stamp}-{uuid.uuid4().hex[:8]}.zip"
        with zipfile.ZipFile(archive_path, "x", compression=zipfile.ZIP_DEFLATED) as archive:
            for path in result_files:
                archive.write(path, arcname=path.relative_to(RESULTS))
        print(f"Download: {archive_path} ({archive_path.stat().st_size / 1024**2:.1f} MiB)")
        display(FileLink(str(archive_path)))
    else:
        print("No result files available to archive.")
